# ScopeOne Demo MMConfig API Walkthrough

This notebook is a broad local API sample using `config/MMConfig_demo.cfg`.

Start `ScopeOne.exe` first. The notebook connects to the running app through the local ScopeOne API server, loads the demo Micro-Manager config, exercises the Python facade, and writes outputs under `examples/output`.


In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
from IPython.display import display

try:
    from PIL import Image
except ImportError:
    Image = None


def add_scopeone_python_path() -> tuple[Path, Path]:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "src" / "scopeone" / "__init__.py").exists() and (base / "pyproject.toml").exists():
            src_dir = base / "src"
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return base, base.parents[2]
    raise RuntimeError("Cannot find ScopeOne python project root")


def to_display_image(frame: np.ndarray) -> np.ndarray:
    frame = np.asarray(frame)
    if frame.dtype == np.uint8:
        return frame
    frame_min = int(frame.min())
    frame_max = int(frame.max())
    if frame_max <= frame_min:
        return np.zeros(frame.shape, dtype=np.uint8)
    scaled = (frame.astype(np.float32) - frame_min) / float(frame_max - frame_min)
    return np.clip(scaled * 255.0, 0, 255).astype(np.uint8)


def show_frame_inline(frame: np.ndarray) -> None:
    print("shape:", frame.shape, "dtype:", frame.dtype, "min/max:", int(frame.min()), int(frame.max()))
    if Image is None:
        print("Pillow is not installed, skipping inline image display")
        return
    display(Image.fromarray(to_display_image(frame)))


project_root, repo_root = add_scopeone_python_path()
config_path = repo_root / "config" / "MMConfig_demo.cfg"
output_dir = project_root / "examples" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

from scopeone import ScopeOne

print("project_root:", project_root)
print("repo_root:", repo_root)
print("config_path:", config_path)
print("output_dir:", output_dir)


## Connect, Load Config, And Inspect Status


In [ ]:
scopeone = ScopeOne.connect("local")
print("version:", scopeone.version())

scopeone.load_config(str(config_path))
status = scopeone.status()
print("status:", status)

camera_ids = scopeone.camera_ids()
loaded_devices = scopeone.loaded_devices()
print("cameras:", camera_ids)
print("loaded devices:", loaded_devices)

if not camera_ids:
    raise RuntimeError("No cameras available after loading MMConfig_demo.cfg")
camera_id = camera_ids[0]
print("using camera:", camera_id)


## Config Groups And Presets


In [ ]:
config_groups = scopeone.config_groups()
original_configs = {}
print("config groups:", config_groups)

for group in config_groups:
    group_configs = scopeone.configs(group)
    current = scopeone.current_config(group)
    original_configs[group] = current
    print(group, "current=", current, "configs=", group_configs)

if "Camera" in config_groups:
    camera_configs = scopeone.configs("Camera")
    if camera_configs:
        target_config = "HighRes" if "HighRes" in camera_configs else camera_configs[0]
        print("set Camera config:", scopeone.set_config("Camera", target_config))
        print("current Camera config:", scopeone.current_config("Camera"))


## Device Properties, Exposure, And ROI


In [ ]:
property_names = scopeone.device_property_names(camera_id)
properties = scopeone.device_properties(camera_id, from_cache=True)
print("property count:", len(property_names))
print("first properties:", property_names[:12])

for item in properties[:8]:
    print(item["name"], "=", item["value"], f"({item['type']})", "readOnly=", item["readOnly"])

if "Exposure" in property_names:
    exposure_property = scopeone.get_property(camera_id, "Exposure", from_cache=False)
    exposure_ms = scopeone.read_exposure(camera_id)
    print("Exposure property:", exposure_property)
    print("Exposure ms:", exposure_ms)
    print("set exposure read-back:", scopeone.set_exposure(exposure_ms, camera_id))
    scopeone.set_property(camera_id, "Exposure", str(exposure_ms))

original_roi = scopeone.get_roi(camera_id)
print("original ROI:", original_roi)
roi_x, roi_y, roi_w, roi_h = original_roi
test_w = max(1, min(roi_w, 256))
test_h = max(1, min(roi_h, 256))
print("set ROI:", scopeone.set_roi(camera_id, roi_x, roi_y, test_w, test_h))
print("read ROI:", scopeone.get_roi(camera_id))
scopeone.clear_roi(camera_id)
print("after clear ROI:", scopeone.get_roi(camera_id))
print("restore ROI:", scopeone.set_roi(camera_id, *original_roi))


## Stage Queries And Safe No-op Moves


In [ ]:
xy_devices = scopeone.xy_stage_devices()
z_devices = scopeone.z_stage_devices()
xy_device = scopeone.current_xy_stage_device()
z_device = scopeone.current_focus_device()
print("XY stages:", xy_devices)
print("Z stages:", z_devices)
print("current XY:", xy_device)
print("current Z:", z_device)

xy0 = None
z0 = None
if xy_device:
    xy0 = scopeone.read_xy_position(xy_device)
    print("XY before:", xy0)
    scopeone.move_xy_relative(0.0, 0.0, xy_device)
    scopeone.move_xy_to(xy0[0], xy0[1], xy_device)
    print("XY after:", scopeone.read_xy_position(xy_device))
if z_device:
    z0 = scopeone.read_z_position(z_device)
    print("Z before:", z0)
    scopeone.move_z_relative(0.0, z_device)
    scopeone.move_z_to(z0, z_device)
    print("Z after:", scopeone.read_z_position(z_device))


## Preview Layers And Markups


In [ ]:
scopeone.start_preview(camera_id)
time.sleep(0.5)

layers = scopeone.list_layers()
options = scopeone.layer_options()
print("layers:", layers)
print("layer options:", options)

layer_key = layers[0]["layerKey"] if layers else f"raw:{camera_id}"
print("using layer:", layer_key)

scopeone.set_layer_layout("overlay")
print("selected layers:", scopeone.set_selected_layers([layer_key]))
print("display:", scopeone.set_layer_display(
    layer_key,
    visible=True,
    opacity_percent=85,
    gamma=1.0,
    colormap="Fire" if "Fire" in options.get("colormaps", []) else None,
    blending="Opaque" if "Opaque" in options.get("blendingModes", []) else None,
    levels=(0, 65535, 65535),
))
print("move layer:", scopeone.move_layer(layer_key, 0))

line_markup_id = scopeone.create_line_markup(layer_key, 5, 5, 128, 128, label="Demo line")
rect_markup_id = scopeone.create_rect_markup(layer_key, 16, 16, 96, 80, label="Demo ROI")
print("markups:", scopeone.list_markups(layer_key))
scopeone.remove_markup(line_markup_id)
scopeone.clear_markups(layer_key)
print("markups after cleanup:", scopeone.list_markups(layer_key))


## Latest Live Frame And Shared Frame Mapping


In [ ]:
live = scopeone.latest_raw_frame(camera_id)
print("FrameResult:", live.camera, live.width, live.height, live.pixel_format, live.bits_per_sample)
print("frame index:", live.frame_index, "timestamp ns:", live.timestamp_ns, "source ROI:", live.source_roi)
show_frame_inline(live.image)

mapping_info = scopeone.frame_mapping_info()
print("frame mapping info:", mapping_info)

synthetic = np.zeros_like(live.image)
synthetic[::8, :] = live.image.max() if live.image.size else 255
synthetic[:, ::8] = live.image.max() if live.image.size else 255
synthetic_layer = scopeone.show_image(
    synthetic,
    layer_id="python_synthetic",
    name="Python Synthetic",
    camera="python_synthetic",
    bits_per_sample=live.bits_per_sample,
)
print("synthetic layer:", synthetic_layer)
mapping_frame = scopeone.write_frame_mapping(
    synthetic,
    camera="python_demo",
    bits_per_sample=live.bits_per_sample,
    frame_index=1,
    source_roi=live.source_roi,
)
mapping_frame.write(mapping_frame.image)
processed_mapping = scopeone.process_frame_mapping(camera="python_demo")
mapping_layer = scopeone.show_frame_mapping_as_layer("python_mapping", "Python Mapping", camera="python_demo")
mapping_paths = scopeone.save_frame_mapping(str(output_dir), "python_mapping", camera="python_demo")
print("processed mapping:", processed_mapping.width, processed_mapping.height)
print("mapping layer:", mapping_layer)
print("mapping saved:", mapping_paths)


## Image Convenience APIs


In [ ]:
shown_image_layer = scopeone.show_image(
    np.flipud(live.image),
    layer_id="python_show_image",
    name="Python show_image",
    camera="python_show_image",
    bits_per_sample=live.bits_per_sample,
)
image_paths = scopeone.save_image(
    np.fliplr(live.image),
    str(output_dir),
    base_name="python_save_image",
    camera="python_save_image",
    bits_per_sample=live.bits_per_sample,
)
print("shown image layer:", shown_image_layer)
print("saved image paths:", image_paths)


## Build A Temporary Processing Pipeline


In [ ]:
scopeone.stop_processing()
scopeone.set_processing_bit_depth(16)
print("processing before:", scopeone.processing_state())

stateful_index = scopeone.add_processing_module("differential_rolling", {"batch_size": 2, "normalize": False})
try:
    scopeone.reset_processing_module_state(stateful_index)
    print("stateful module reset:", stateful_index)
finally:
    scopeone.remove_processing_module(stateful_index)

module_index = scopeone.add_processing_module("gaussian_blur", {"kernel_size": 5, "sigma": 1.2})
try:
    print("added module index:", module_index)
    print("modules:", scopeone.processing_modules())
    scopeone.set_processing_module_parameters(module_index, {"kernel_size": 7, "sigma": 1.0})

    stage = scopeone.process_image(
        live.image,
        camera="python_pipeline",
        bits_per_sample=live.bits_per_sample,
        end_module_index=module_index,
    )
    result = scopeone.continue_pipeline(stage, image=stage.image)
    result_layer = scopeone.show_frame(result, layer_id="python_pipeline_result", name="Python Pipeline Result")
    result_paths = scopeone.save_frame(result, str(output_dir), base_name="python_pipeline_result")
    print("result layer:", result_layer)
    print("result saved:", result_paths)

    scopeone.start_processing()
    time.sleep(0.5)
    print("processing real-time:", scopeone.processing_state())
finally:
    scopeone.stop_processing()
    scopeone.remove_processing_module(module_index)
    print("processing after cleanup:", scopeone.processing_state())


## Recording Session APIs


In [ ]:
with scopeone.record(frames=3, camera=camera_id) as session:
    print("session cameras:", session.camera_ids())
    print("session frame count:", session.frame_count(camera_id))
    session_frame = session.frame(camera_id, 0)
    session_processed = session.process_frame(camera_id, 0)
    session_frames = session.frames(camera_id)
    session_paths = session.save(str(output_dir), base_name="demo_session", format="tiff")
    print("session frame:", session_frame.width, session_frame.height)
    print("session processed:", session_processed.width, session_processed.height)
    print("session frames loaded:", len(session_frames))
    print("session saved:", session_paths)
show_frame_inline(session_frame.image)


## Minimal MDA Recording With Demo Stages


In [ ]:
mda_positions = None
if xy0 is not None:
    mda_positions = [xy0]

mda_z = [z0, z0] if z0 is not None else None
if mda_z is None and mda_positions is None:
    print("No demo stage devices available, skipping MDA stage path")
else:
    with scopeone.record(
        frames=2,
        camera=camera_id,
        mda_interval_ms=0.0,
        z_positions=mda_z,
        positions=mda_positions,
        order=["time", "z", "xy"],
    ) as mda_session:
        print("MDA cameras:", mda_session.camera_ids())
        print("MDA frame count:", mda_session.frame_count(camera_id))
        mda_paths = mda_session.save(str(output_dir), base_name="demo_mda", format="tiff")
        print("MDA saved:", mda_paths)

if xy0 is not None:
    scopeone.move_xy_to(xy0[0], xy0[1], xy_device)
if z0 is not None:
    scopeone.move_z_to(z0, z_device)


## Cleanup


In [ ]:
for candidate in [
    globals().get("synthetic_layer"),
    globals().get("mapping_layer"),
    globals().get("shown_image_layer"),
    globals().get("result_layer"),
]:
    if candidate:
        try:
            scopeone.remove_static_layer(candidate)
        except RuntimeError as exc:
            print("remove_static_layer skipped:", candidate, exc)

scopeone.clear_static_layers()
scopeone.clear_markups()
scopeone.set_layer_layout("side_by_side")

for group, config in original_configs.items():
    if config:
        try:
            scopeone.set_config(group, config)
        except RuntimeError as exc:
            print("restore config skipped:", group, config, exc)

scopeone.stop_preview(camera_id)
scopeone.unload_config()
scopeone.close()
print("cleanup complete")
